In [1]:
import math
import tqdm
import string
import time
import random
import numpy as np
import pandas as pd
import re, os, random
import matplotlib.pyplot as plt

import pickle
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Set device based on GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
torch.manual_seed(10701)
random.seed(10701)
np.random.seed(10701)

In [ ]:
AA_VOCAB  = list('ACDEFGHIKLMNPQRSTVWXY')
SS8_VOCAB = ['C', 'B', 'E', 'G', 'I', 'H', 'S', 'T']
MAX_LEN   = 700

class ProteinDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.df      = df.reset_index(drop=True)
        self.max_len = max_len
        self.aa_map  = {c: i for i, c in enumerate(AA_VOCAB)}
        self.ss_map  = {c: i for i, c in enumerate(SS8_VOCAB)}

    def __len__(self):
        return len(self.df)

    def encode(self, seq, vocab_map):
        arr = np.zeros(self.max_len, dtype=np.int64)
        for i, ch in enumerate(seq[:self.max_len]):
            if ch in vocab_map:
                arr[i] = vocab_map[ch]
        return arr

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        X      = self.encode(row['input'], self.aa_map)
        y      = self.encode(row['dssp8'], self.ss_map)
        length = min(len(row['input']), self.max_len)
        mask   = np.zeros(self.max_len, dtype=np.float32)
        mask[:length] = 1.0
        return (torch.tensor(X),
                torch.tensor(y),
                torch.tensor(mask))

In [3]:
train_df = pd.read_csv('/content/drive/My Drive/2025-26/10701/train_data.csv')
test_df = pd.read_csv('/content/drive/My Drive/2025-26/10701/test_data.csv')

train_dataset = ProteinDataset(train_df)
test_dataset  = ProteinDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

NameError: name 'ProteinDataset' is not defined

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size, output_size, batch_size):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.batch_size = batch_size

        # Initialize embeddings, model layers, layer normalization, and activation function
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=input_size, padding_idx=0)
        self.input_to_hidden = torch.nn.Linear(input_size, hidden_size)
        self.hidden_to_hidden = torch.nn.Linear(hidden_size, hidden_size)
        self.hidden_to_out = torch.nn.Linear(hidden_size, output_size)

        self.layer_norm = torch.nn.LayerNorm(hidden_size)
        self.tanh = torch.nn.Tanh()
        self.softmax = torch.nn.Softmax()


    def forward(self, input: torch.Tensor, hidden: torch.Tensor) -> torch.Tensor:
        # Implement forward pass including layer normalization

        h = hidden
        e = self.embedding(input)
        y = []
        for t in range(e.shape[1]):
            x_t = e[:, t, :]
            h = self.tanh(self.layer_norm(self.input_to_hidden(x_t) + self.hidden_to_hidden(h)))
            y_t = self.softmax(self.hidden_to_out(h))
            y.append(y_t)

        return torch.stack(y, dim=1)

    def init_hidden(self) -> torch.Tensor:
        # Initialize hidden state to zeros
        # print(f"Batch size: {self.batch_size}, Hidden size: {self.hidden_size}")
        h0 = torch.zeros(self.batch_size, self.hidden_size)
        return h0

In [ ]:
def train(data_loader, model, criterion, optimizer):
    test_avg_loss = 0
    num_correct = 0

    total_samples = 0

    # Set model to training mode
    model.train()

    for samples, labels in tqdm.tqdm(data_loader, leave=False):
        # Convert labels to LongTensors and move to device
        labels = labels.long().to(device)
        # labels = labels.to(device)

        # samples = torch.tensor(samples).to(device)
        samples = samples.to(device)

        total_samples += labels.size(0)

        # Initialize hidden state
        h0 = model.init_hidden().to(device)

        # print(f"Labels dtype = {labels.dtype}, Samples dtype = {samples.dtype}, h0 dtype = {h0.dtype}")

        # Perform forward pass and calculate avg loss over all time steps
        out = model.forward(samples, h0)

        num_tokens = out.shape[1]
        loss = 0
        for t in range(out.shape[1]):
            loss += criterion(out[:, t, :], labels)
        # loss = criterion(out, labels)
        preds = torch.argmax(out[:, -1, :], dim=-1)
        test_avg_loss += loss.item() * labels.size(0)

        # Calculate number of correct predictions
        acc = torch.sum(preds == labels)
        num_correct += acc

        # Backward pass, update weights, and zero gradients
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Uncomment the line below once the training loop is implemented to print
    # the loss and accuracy at the end of each epoch

    total_samples_loss = total_samples * num_tokens
    print(f"Total Samples = {total_samples}, len(data_loader) = {len(data_loader)}")
    print(f"Train loss: {test_avg_loss/total_samples_loss} | Accuracy: {num_correct/total_samples}")
    train_loss = test_avg_loss/total_samples_loss
    train_acc = num_correct/total_samples
    return train_loss, train_acc

In [ ]:
def test(data_loader, rnn, criterion):
    test_avg_loss = 0
    num_correct = 0

    total_samples = 0

    # Set model to eval mode
    rnn.eval()

    # Implement testing loop
    # Hint: Do we want to update gradients when testing?
    with torch.no_grad():
        for samples, labels in tqdm.tqdm(data_loader, leave=False):
            # Convert labels to LongTensors and move to device
            labels = labels.long().to(device)

            # Initialize hidden state
            h0 = rnn.init_hidden().to(device)

            samples = samples.to(device)

            total_samples += labels.size(0)

            # Perform forward pass and calculate avg loss over all time steps
            out = rnn.forward(samples, h0)
            preds = torch.argmax(out[:, -1, :], dim=-1)

            num_tokens = out.shape[1]
            loss = 0
            for t in range(out.shape[1]):
                loss += criterion(out[:, t, :], labels)

            test_avg_loss += loss * labels.size(0)

            # Calculate number of correct predictions
            acc = torch.sum(preds == labels)
            num_correct += acc

        total_samples_loss = total_samples * num_tokens
        # Uncomment the line below once the training loop is implemented to print
        # the loss and accuracy at the end of each epoch
        print(f"Test loss: {test_avg_loss/total_samples_loss} | Accuracy: {num_correct/total_samples}")


        test_loss = test_avg_loss / total_samples_loss
        test_acc = num_correct / total_samples
        return test_loss, test_acc

In [ ]:
def run(num_epochs, train_dataloader, test_dataloader, rnn, criterion, optimizer):
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    for epoch in range(num_epochs):
        print(f"Epoch {epoch}")
        # Perform one epoch of training and testing
        train_loss, train_acc = train(train_dataloader, rnn, criterion, optimizer)
        test_loss, test_acc = test(test_dataloader, rnn, criterion)

        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)

    return train_losses, test_losses, train_accs, test_accs

In [ ]:
def main(
    # Hyperparameters
    vocabulary_size = 20000,
    batch_size = 128,
    input_size = 64,
    hidden_size = 128,
    max_review_length = MAX_LEN,
    lr = 1e-4,
    num_epochs = 10,
    num_classes = 2
):
    input_size = MAX_LEN
    num_train_samples = 10790
    num_test_samples = 432
    AA_vocab_size = len(AA_VOCAB) # 21
    SS8_vocab_size = len(SS8_VOCAB) # 8
    num_classes = 3 # using SS, not SS8?

    # Initialize model
    model = RNN(AA_vocab_size, MAX_LEN, hidden_size, num_classes, batch_size)
    model.to(device)

    # Initialize loss function and optimizer
    loss_fn = torch.nn.CrossEntropyLoss()
    optim = torch.optim.Adam(model.parameters(), lr)

    # Run training and testing
    train_losses, test_losses, train_accs, test_accs = run(num_epochs, train_dataloader, test_dataloader, model, loss_fn, optim)
    # run(num_epochs, train_dataloader, test_dataloader, model, loss_fn, optim)

    # print(f"Train losses = {train_losses}")
    # print(f"Test losses = {test_losses}")
    return train_losses, test_losses

In [ ]:
main()